# GPT-2 Training for Vietnamese Nguyễn Du Poetry Generation


Complete the functions/snipshot marked with ## TODO.

After completing this exercise, stundent are learnt to implement Vietnamese text normalization and syllable counting for preprocessing poetry data, create structured datasets by pairing lines into lục bát couplets (6-8 syllable meter) with special control tokens, initialize and configure GPT-2 models with custom special tokens and tokenizers for Vietnamese, tokenize datasets and set up training pipelines using the Hugging Face Transformers Trainer API with appropriate hyperparameters, train causal language models on Vietnamese poetry data while managing memory constraints, and evaluate generated poems by validating whether they follow the traditional lục bát poetic structure through programmatic checking.

In [1]:
# !pip install datasets
# !pip install transformers[torch]

In [2]:

# Import Required Libraries
import os
import re
import random
from typing import List, Tuple, Dict

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Libraries imported successfully!
PyTorch version: 2.4.1+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3090


In [3]:
# Configuration and Constants
MODEL_NAME = "gpt2"
DATA_PATH = "dataset/truyenkieu.txt" 
OUT_DIR = "./gpt2_kieu_lucbat"

# Training hyperparameters (optimized for low memory usage)
SEED = 42
MAX_LEN = 128
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 10  # Start with 1 epoch for testing
LR = 5e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
VAL_RATIO = 0.05

# Set random seed for reproducibility
set_seed(SEED)

print("Configuration set:")
print(f"Model: {MODEL_NAME}")
print(f"Data path: {DATA_PATH}")
print(f"Output directory: {OUT_DIR}")
print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Configuration set:
Model: gpt2
Data path: dataset/truyenkieu.txt
Output directory: ./gpt2_kieu_lucbat
Training epochs: 10
Batch size: 1
Device: CUDA


In [ ]:
# Text Preprocessing Functions for Vietnamese

import unicodedata

# Regular expression to keep Vietnamese letters, spaces, hyphens, and apostrophes
_punct_re = re.compile(r"[^\wÀ-ỹ\s'-]+", flags=re.UNICODE)

def normalize_line(line: str) -> str:
    """Normalize Vietnamese text line for syllable counting."""
    if line is None:
        return ""
    # Unicode NFC so that composed/decomposed Vietnamese diacritics are identical
    line = unicodedata.normalize("NFC", line)
    line = line.lower()
    # Remove punctuation (keep letters, digits, spaces, hyphens, apostrophes)
    line = _punct_re.sub(" ", line)
    # Collapse multiple whitespace and trim
    line = re.sub(r"\s+", " ", line).strip()
    return line

def count_syllables_approx_vn(line: str) -> int:    
    """
    Approximate Vietnamese syllable count by counting whitespace-separated words.
    This is a simple but effective method for Vietnamese.
    """
    line = normalize_line(line)
    if not line:
        return 0
    return len(line.split())

def looks_like_lucbat_pair(l1: str, l2: str, tol: int = 0) -> bool:
    """
    Check if two lines form a lục bát couplet (6-8 syllables).
    Allows tolerance for slight variations after normalization.
    """
    s1 = count_syllables_approx_vn(l1)
    s2 = count_syllables_approx_vn(l2)
    return (6 - tol <= s1 <= 6 + tol) and (8 - tol <= s2 <= 8 + tol)

# Test the functions
test_line1 = "Trăm năm trong cõi người ta"  
test_line2 = "Chữ tài chữ mệnh khéo là ghét nhau"  

print("Testing Vietnamese text processing:")
print(f"Line 1: '{test_line1}'")
print(f"Syllables: {count_syllables_approx_vn(test_line1)}")
print(f"Line 2: '{test_line2}'")
print(f"Syllables: {count_syllables_approx_vn(test_line2)}")
print(f"Is Lục Bát pair: {looks_like_lucbat_pair(test_line1, test_line2)}")

In [ ]:
# Dataset Preparation Functions

# Special control tokens
BOS_TOKEN = "<KIEU>"
LB_OPEN = "<LB>"
LB_CLOSE = "</LB>"
SPECIAL_TOKENS = [BOS_TOKEN, LB_OPEN, LB_CLOSE]

def load_lines(path: str) -> List[str]:
    """Load and preprocess text lines from file."""
    assert os.path.exists(path), f"Missing file: {path}"
    
    with open(path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f]
    
    # Remove empty lines and normalize
    lines = [ln for ln in lines if ln.strip()]
    lines = [normalize_line(ln) for ln in lines]
    lines = [ln for ln in lines if ln]
    
    return lines

def make_couplet_samples(lines: List[str], strict: bool = True) -> List[str]:
    """
    Convert line list into training samples with special control tokens.
    Each sample is a 2-line couplet wrapped with <KIEU><LB>...</LB> tokens.
    """
    samples = []
    i = 0
    
    while i < len(lines) - 1:
        l1, l2 = lines[i], lines[i + 1]
        if strict and not looks_like_lucbat_pair(l1, l2):
            # Misaligned / irregular line: skip one line and try to re-sync
            i += 1
            continue
        samples.append(f"{BOS_TOKEN}{LB_OPEN}\n{l1}\n{l2}\n{LB_CLOSE}")
        i += 2
    
    return samples

def prepare_dataset(data_path: str) -> Dataset:
    """Prepare training dataset from text file."""
    print(f"Loading text from: {data_path}")
    
    # Load and preprocess lines
    raw_lines = load_lines(data_path)
    print(f"Loaded {len(raw_lines)} lines")
    print(f"First 3 lines: {raw_lines[:3]}")
    
    # Create couplet samples
    samples = make_couplet_samples(raw_lines, strict=True)
    print(f"Created {len(samples)} couplet samples")
    print(f"First training sample:\n{samples[0]}")
    
    # Shuffle and split
    random.Random(SEED).shuffle(samples)
    ds = Dataset.from_dict({"text": samples})
    ds = ds.train_test_split(test_size=VAL_RATIO, seed=SEED)
    
    return ds

# Load and prepare the dataset
dataset = prepare_dataset(DATA_PATH)
print(f"\nDataset structure: {dataset}")

In [ ]:
# Model and Tokenizer Setup

def setup_model_and_tokenizer():
    """Initialize tokenizer and model with special tokens."""
    print(f"Loading model: {MODEL_NAME}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Add special tokens for style control
    num_added = tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    print(f"Added {num_added} special tokens: {SPECIAL_TOKENS}")
    
    # Set pad token (GPT-2 doesn't have one by default)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Determine device and model precision
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Keep weights in fp32; mixed precision (fp16) is handled by the Trainer on GPU
    dtype = torch.float32
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
    model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(device)
    print(f"Parameters: {model.num_parameters()/1e6:.1f}M, vocab size: {len(tokenizer)}")
    
    print(f"Device: {device}")
    
    return model, tokenizer, device

# Initialize model and tokenizer
model, tokenizer, device = setup_model_and_tokenizer()

In [ ]:
# Training Pipeline
import math

def train_model(model, tokenizer, dataset):
    """Train the model on the dataset."""
    
    def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
        return tokenizer(
            [t + tokenizer.eos_token for t in batch["text"]],
            truncation=True,
            max_length=MAX_LEN,
        )
    
    tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
    print(tokenized)
    
    # Causal LM collator: pads dynamically and builds labels (= input_ids, pad -> -100)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=OUT_DIR,
        overwrite_output_dir=True,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LR,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to="none",
    )
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        data_collator=data_collator,
    )
    
    print("Starting training...")
    
    # Train the model
    trainer.train()
    
    metrics = trainer.evaluate()
    print(f"Eval loss: {metrics['eval_loss']:.4f} | perplexity: {math.exp(metrics['eval_loss']):.2f}")
    
    # Save model and tokenizer (if you want to)
    trainer.save_model(OUT_DIR)
    tokenizer.save_pretrained(OUT_DIR)
    print(f"Model saved to: {OUT_DIR}")
    
    return trainer

# Start training
trainer = train_model(model, tokenizer, dataset)

In [ ]:
# Evaluation: generate poems and check the lục bát structure

@torch.no_grad()
def generate_poem(prompt_first_line: str = "", max_new_tokens: int = 60,
                  temperature: float = 0.9, top_p: float = 0.95) -> str:
    """Generate one couplet. Optionally seed it with the beginning of the lục line."""
    model.eval()
    prompt = f"{BOS_TOKEN}{LB_OPEN}\n"
    if prompt_first_line:
        prompt += normalize_line(prompt_first_line)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.convert_tokens_to_ids(LB_CLOSE),
    )
    return tokenizer.decode(out[0], skip_special_tokens=False)

def extract_couplet(text: str) -> Tuple[str, str]:
    """Extract the two lines between <LB> and </LB>."""
    body = text.split(LB_OPEN, 1)[-1].split(LB_CLOSE, 1)[0]
    body = body.replace(tokenizer.eos_token, "")
    lines = [ln.strip() for ln in body.strip().split("\n") if ln.strip()]
    l1 = lines[0] if len(lines) > 0 else ""
    l2 = lines[1] if len(lines) > 1 else ""
    return l1, l2

set_seed(SEED)
prompts = ["", "", "", "", "", "trăm năm", "tình duyên", "đêm khuya", "người về", "hoa trôi"]
n_ok = 0
for k, p in enumerate(prompts, 1):
    l1, l2 = extract_couplet(generate_poem(p))
    ok = looks_like_lucbat_pair(l1, l2)
    n_ok += ok
    print(f"[{k:02d}] prompt={p!r}")
    print(f"     {l1}   ({count_syllables_approx_vn(l1)} âm tiết)")
    print(f"     {l2}   ({count_syllables_approx_vn(l2)} âm tiết)")
    print(f"     -> lục bát: {'✔' if ok else '✘'}")

print(f"\nLục bát structure accuracy: {n_ok}/{len(prompts)} = {n_ok/len(prompts):.0%}")
